In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy import stats
from scipy.signal import correlate

merged = pd.read_csv("../data/processed/all_cities_combined.csv")
merged["week"] = pd.to_datetime(merged["week"])

CITIES = ["bangkok", "krakow", "melbourne", "lahore"]
TERMS  = ["air_quality", "air_purifier", "N95_mask"]
HEALTH_TERMS = ["hospital", "breathing_difficulty", "cough_medicine", "eye_drops", "chest_pain"]
ALL_TERMS = TERMS + HEALTH_TERMS

print("Loaded ✓")
print(merged.groupby("city")[["pm25_mean","air_quality"]].describe().round(2))

In [ ]:
def assign_bins(df):
    df = df.copy()
    df["pm25_bin"] = pd.qcut(df["pm25_mean"], q=3,
                              labels=["Low", "Medium", "High"])
    return df

fig, axes = plt.subplots(1, len(CITIES), figsize=(5.3 * len(CITIES), 6))

for ax, city in zip(axes, CITIES):
    df = merged[merged["city"] == city].copy().dropna()
    df = assign_bins(df)

    data_by_bin = [df[df["pm25_bin"]==b]["air_quality"].values
                   for b in ["Low", "Medium", "High"]]

    bp = ax.boxplot(data_by_bin, labels=["Low\nPM2.5", "Medium\nPM2.5", "High\nPM2.5"],
                    patch_artist=True, notch=False,
                    boxprops=dict(facecolor="#d6e4f0"),
                    medianprops=dict(color="#d62728", linewidth=2))

    for i, d in enumerate(data_by_bin, 1):
        ax.scatter(np.random.normal(i, 0.07, len(d)), d,
                   alpha=0.4, s=15, color="#2c7bb6", zorder=3)

    f, p = stats.f_oneway(*data_by_bin)
    sig = f"p={p:.4f} ✓" if p < 0.05 else f"p={p:.4f} ✗"

    ax.set_title(f"{city.capitalize()}\nANOVA: F={f:.2f}, {sig}",
                 fontweight="bold")
    ax.set_ylabel("Search Index — 'air quality'")
    ax.set_xlabel("PM2.5 Pollution Level")
    ax.grid(axis="y", alpha=0.3)

plt.suptitle("Search Interest by PM2.5 Level (Tertile Bins)",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("../outputs/figures/boxplot_bins.png", dpi=150, bbox_inches="tight")
plt.show()
print("✓ Saved → outputs/figures/boxplot_bins.png")

In [ ]:
fig, axes = plt.subplots(1, len(CITIES), figsize=(5.3 * len(CITIES), 5))

for ax, city in zip(axes, CITIES):
    df = merged[merged["city"] == city].dropna()
    x = df["pm25_mean"]
    y = df["air_quality"]

    ax.scatter(x, y, alpha=0.4, s=25, color="#2c7bb6", zorder=3)

    slope, intercept, r, p, se = stats.linregress(x, y)
    x_line = np.linspace(x.min(), x.max(), 100)
    ax.plot(x_line, slope * x_line + intercept,
            color="#d62728", linewidth=2,
            label=f"r={r:.3f}, p={p:.4f}")

    n = len(x)
    x_mean = x.mean()
    se_band = se * np.sqrt(1/n + (x_line - x_mean)**2 / ((x - x_mean)**2).sum())
    t_crit = stats.t.ppf(0.975, df=n-2)
    y_pred = slope * x_line + intercept
    ax.fill_between(x_line,
                    y_pred - t_crit * se_band,
                    y_pred + t_crit * se_band,
                    alpha=0.15, color="#d62728")

    ax.set_title(f"{city.capitalize()}", fontweight="bold")
    ax.set_xlabel("PM2.5 (µg/m³)")
    ax.set_ylabel("Search Index — 'air quality'")
    ax.legend(fontsize=9)
    ax.grid(alpha=0.3)

plt.suptitle("PM2.5 vs Search Interest — Linear Regression per City",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("../outputs/figures/scatter_regression.png", dpi=150, bbox_inches="tight")
plt.show()
print("✓ Saved → outputs/figures/scatter_regression.png")

In [ ]:
fig, axes = plt.subplots(1, len(CITIES), figsize=(3.4 * len(CITIES), 6),
                         sharey=True, layout="constrained")

for ax, city in zip(axes, CITIES):
    df = merged[merged["city"] == city].dropna()
    r_vals = []
    for term in ALL_TERMS:
        if df[term].std() == 0:
            r_vals.append(np.nan)
            continue
        r, p = stats.pearsonr(df["pm25_mean"], df[term])
        r_vals.append(r)

    ax.set_facecolor("#e6e6e6")
    im = ax.imshow(np.array(r_vals).reshape(-1, 1), cmap="RdBu_r",
                   vmin=-1, vmax=1, aspect="auto")

    for i, r in enumerate(r_vals):
        ax.text(0, i, "n/a" if np.isnan(r) else f"{r:+.2f}",
                ha="center", va="center", fontsize=11,
                color="white" if abs(r) > 0.6 else "black")

    ax.set_xticks([])
    ax.set_yticks(range(len(ALL_TERMS)))
    ax.set_yticklabels(ALL_TERMS)
    ax.set_title(f"{city.capitalize()}", fontweight="bold")

fig.colorbar(im, ax=axes, label="Pearson r", shrink=0.8)
fig.suptitle("Pearson Correlation: PM2.5 vs Search Terms",
             fontsize=13, fontweight="bold")
plt.savefig("../outputs/figures/correlation_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()
print("✓ Saved → outputs/figures/correlation_heatmap.png")

In [ ]:
WINDOW = 12
MIN_PERIODS = 8

city_line_colors = {
    "bangkok":   "#1f77b4",
    "krakow":    "#2ca02c",
    "melbourne": "#9467bd",
    "lahore":    "#ff7f0e"
}

def rolling_pearson(df, window=WINDOW, min_periods=MIN_PERIODS):
    def window_r(positions):
        part = df.iloc[positions.astype(int)][["pm25_mean", "air_quality"]].dropna()
        if len(part) < min_periods or part["pm25_mean"].std() == 0 or part["air_quality"].std() == 0:
            return np.nan
        return stats.pearsonr(part["pm25_mean"], part["air_quality"])[0]

    positions = pd.Series(np.arange(len(df)), index=df.index, dtype=float)
    return positions.rolling(window, min_periods=min_periods).apply(window_r, raw=True)

fig, ax = plt.subplots(figsize=(14, 5))

for city in CITIES:
    df = merged[merged["city"] == city].sort_values("week").set_index("week").asfreq("W-SUN")
    rolling_r = rolling_pearson(df)
    ax.plot(rolling_r.index, rolling_r.values, color=city_line_colors[city],
            linewidth=1.8, label=city.capitalize())

ax.axhline(0, color="black", linestyle="--", linewidth=1, alpha=0.6)
ax.set_ylim(-1.05, 1.05)
ax.set_title(f"Rolling {WINDOW}-Week Pearson Correlation: PM2.5 vs 'air quality' Search",
             fontsize=13, fontweight="bold")
ax.set_ylabel("Pearson r")
ax.set_xlabel("Week")
ax.legend(fontsize=9)
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig("../outputs/figures/rolling_correlation.png", dpi=150, bbox_inches="tight")
plt.show()
print("✓ Saved → outputs/figures/rolling_correlation.png")

In [ ]:
THRESHOLD = 25

fig, axes = plt.subplots(1, len(CITIES), figsize=(5.3 * len(CITIES), 5))

for ax, city in zip(axes, CITIES):
    df = merged[merged["city"] == city].dropna()
    below = df[df["pm25_mean"] < THRESHOLD]
    above = df[df["pm25_mean"] >= THRESHOLD]

    for part, color, name in [(below, "#1f77b4", "Below"), (above, "#d62728", "Above")]:
        ax.scatter(part["pm25_mean"], part["air_quality"],
                   alpha=0.35, s=25, color=color, zorder=3)
        if len(part) < 3 or part["pm25_mean"].nunique() < 2:
            continue
        slope, intercept, r, p, se = stats.linregress(part["pm25_mean"], part["air_quality"])
        x_line = np.linspace(part["pm25_mean"].min(), part["pm25_mean"].max(), 100)
        ax.plot(x_line, slope * x_line + intercept,
                color=color, linewidth=2,
                label=f"{name}: slope={slope:.2f}, r={r:.2f}, n={len(part)}")

    ax.axvline(THRESHOLD, color="black", linestyle="--", linewidth=1,
               label=f"WHO guideline ({THRESHOLD} µg/m³)")
    ax.set_title(f"{city.capitalize()}", fontweight="bold")
    ax.set_xlabel("PM2.5 (µg/m³)")
    ax.set_ylabel("Search Index — 'air quality'")
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)

plt.suptitle("Threshold Regression: Search Response Below vs Above WHO Guideline",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("../outputs/figures/threshold_regression.png", dpi=150, bbox_inches="tight")
plt.show()
print("✓ Saved → outputs/figures/threshold_regression.png")

In [ ]:
MAX_LAG = 8

fig, axes = plt.subplots(len(CITIES), 1, figsize=(11, 3.2 * len(CITIES)), sharex=True)

for ax, city in zip(axes, CITIES):
    df = merged[merged["city"] == city].dropna().copy()
    pm25_norm = (df["pm25_mean"] - df["pm25_mean"].mean()) / df["pm25_mean"].std()

    peak_lags = []
    for term in ALL_TERMS:
        series = df[term]
        if series.std() == 0:
            peak_lags.append(np.nan)
            continue
        srch_norm = (series - series.mean()) / series.std()
        cc = correlate(srch_norm, pm25_norm, mode="full")
        lags = np.arange(-(len(pm25_norm)-1), len(pm25_norm))
        mask = (lags >= -MAX_LAG) & (lags <= MAX_LAG)
        peak_lags.append(lags[mask][np.argmax(cc[mask])])

    colors = ["#2ca02c" if lag < 0 else "#d62728" if lag > 0 else "grey"
              for lag in peak_lags]
    ax.barh(ALL_TERMS, np.nan_to_num(peak_lags), color=colors, edgecolor="white")

    for i, lag in enumerate(peak_lags):
        label = "n/a" if np.isnan(lag) else f"{int(lag):+d}"
        offset = 0.3 if np.isnan(lag) or lag >= 0 else -0.3
        ax.text(np.nan_to_num(lag) + offset, i, label, va="center",
                ha="left" if offset > 0 else "right", fontsize=9,
                color="grey" if np.isnan(lag) or lag == 0 else "black")

    ax.axvline(0, color="black", linestyle="--", linewidth=0.8, alpha=0.5)
    ax.set_xlim(-MAX_LAG - 2, MAX_LAG + 2)
    ax.set_xticks(range(-MAX_LAG, MAX_LAG + 1))
    ax.set_title(f"{city.capitalize()}", fontweight="bold")
    ax.grid(axis="x", alpha=0.3)

axes[-1].set_xlabel("Peak lag (weeks)\n← search leads | pollution leads →")
plt.suptitle("Peak Cross-Correlation Lag by Search Term\n(green = search leads PM2.5, red = search lags PM2.5, grey = simultaneous)",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("../outputs/figures/term_lag_comparison.png", dpi=150, bbox_inches="tight")
plt.show()
print("✓ Saved → outputs/figures/term_lag_comparison.png")

In [ ]:
HORIZONS = [1, 2, 3]

fig, axes = plt.subplots(1, len(CITIES), figsize=(4.5 * len(CITIES), 5))

for ax, city in zip(axes, CITIES):
    df = merged[merged["city"] == city].sort_values("week").set_index("week").asfreq("W-SUN")
    change = df["pm25_mean"].diff()
    rising = change > 0
    falling = change < 0

    rise_means = [df["air_quality"].shift(-h)[rising].mean() for h in HORIZONS]
    fall_means = [df["air_quality"].shift(-h)[falling].mean() for h in HORIZONS]

    x = np.arange(len(HORIZONS))
    width = 0.38
    ax.bar(x - width/2, rise_means, width, color="#d62728",
           label=f"After rising PM2.5 (n={int(rising.sum())})")
    ax.bar(x + width/2, fall_means, width, color="#1f77b4",
           label=f"After falling PM2.5 (n={int(falling.sum())})")

    ax.set_xticks(x)
    ax.set_xticklabels([f"+{h}w" for h in HORIZONS])
    ax.set_title(f"{city.capitalize()}", fontweight="bold")
    ax.set_xlabel("Weeks after PM2.5 change")
    ax.set_ylabel("Mean Search Index — 'air quality'")
    ax.legend(fontsize=8)
    ax.grid(axis="y", alpha=0.3)

plt.suptitle("Lag Asymmetry: Search Response After Rising vs Falling PM2.5",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("../outputs/figures/lag_asymmetry.png", dpi=150, bbox_inches="tight")
plt.show()
print("✓ Saved → outputs/figures/lag_asymmetry.png")

In [ ]:
YEARS = [2022, 2023, 2024]
MIN_ROWS = 20

yoy = pd.DataFrame(index=CITIES, columns=YEARS, dtype=float)

for city in CITIES:
    df = merged[merged["city"] == city].dropna()
    for year in YEARS:
        part = df[df["week"].dt.year == year]
        if len(part) < MIN_ROWS or part["pm25_mean"].std() == 0 or part["air_quality"].std() == 0:
            continue
        r, p = stats.pearsonr(part["pm25_mean"], part["air_quality"])
        yoy.loc[city, year] = r

print("=== YEAR-OVER-YEAR PEARSON r: PM2.5 vs 'air quality' ===\n")
print(yoy.round(3))

fig, ax = plt.subplots(figsize=(12, 5))

x = np.arange(len(CITIES))
width = 0.8 / len(YEARS)
for i, (year, color) in enumerate(zip(YEARS, ["#9ecae1", "#4292c6", "#084594"])):
    ax.bar(x + (i - (len(YEARS)-1)/2) * width, yoy[year].values, width,
           color=color, edgecolor="white", label=str(year))

ax.axhline(0, color="black", linestyle="-", linewidth=0.5, alpha=0.5)
ax.set_xticks(x)
ax.set_xticklabels([city.capitalize() for city in CITIES])
ax.set_title("Year-over-Year Pearson r: PM2.5 vs 'air quality' Search",
             fontsize=13, fontweight="bold")
ax.set_ylabel("Pearson r")
ax.set_xlabel("City")
ax.legend(title="Year", fontsize=9)
ax.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.savefig("../outputs/figures/yoy_correlation.png", dpi=150, bbox_inches="tight")
plt.show()
print("✓ Saved → outputs/figures/yoy_correlation.png")

In [ ]:
print("=" * 65)
print(f"{'FINAL RESULTS SUMMARY':^65}")
print("=" * 65)

rows = []
for city in CITIES:
    df = merged[merged["city"] == city].dropna()
    for term in ALL_TERMS:
        if df[term].std() == 0:
            continue
        r, p = stats.pearsonr(df["pm25_mean"], df[term])
        pm25_norm = (df["pm25_mean"] - df["pm25_mean"].mean()) / df["pm25_mean"].std()
        srch_norm = (df[term] - df[term].mean()) / df[term].std()
        cc = correlate(srch_norm, pm25_norm, mode="full")
        lags = np.arange(-(len(pm25_norm)-1), len(pm25_norm))
        mask = (lags >= -8) & (lags <= 8)
        peak_lag = lags[mask][np.argmax(cc[mask])]
        rows.append({
            "City": city.capitalize(),
            "Term": term,
            "Pearson r": round(r, 3),
            "p-value": round(p, 4),
            "Significant": "✓" if p < 0.05 else "✗",
            "Peak Lag (weeks)": int(peak_lag)
        })

summary = pd.DataFrame(rows)
print(summary.to_string(index=False))
summary.to_csv("../data/processed/results_summary.csv", index=False)
print("\n✓ Saved → data/processed/results_summary.csv")